In [1]:
import numpy as np
import xml.etree.ElementTree as ET
import napari
import tifffile as tiff
import scimap as sm
import anndata as ad
import pandas as pd
import os

Running SCIMAP  2.3.5


/Users/lukashat/miniforge3/envs/scimap_benchmark/lib/python3.10/site-packages/mpl_scatter_density/__init__.py:4: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.

/Users/lukashat/miniforge3/envs/scimap_benchmark/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



In [2]:
def print_metadata_structure(element, indent=0):
    print("  " * indent + f"Tag: {element.tag}")
    if element.attrib:
        print("  " * (indent + 1) + "Attributes:")
        for key, value in element.attrib.items():
            print("  " * (indent + 2) + f"{key}: {value}")
    for child in element:
        print_metadata_structure(child, indent + 1)

In [15]:
dataset = 'Breast_Jun0524'

In [18]:
dataset_path_string = 'ManualLabels/Breast_Jun0524'

In [12]:
sds_dir = "/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/"

In [30]:
ome_path = os.path.join(sds_dir, f'{dataset_path_string}/raw_images/multistack_tiffs/B-12.ome.tif')
subset = 'B-12'
seg_path = os.path.join(sds_dir, f'{dataset_path_string}/segmentation/B-12.tiff')

In [19]:
quant = pd.read_csv(os.path.join(sds_dir, f'{dataset_path_string}/quantification/processed/{dataset}_quantification.csv'))
#img = tiff.imread(ome_path)
with open(os.path.join(sds_dir, f'{dataset_path_string}/markers.txt')) as f:
    channel_names = [line.strip() for line in f.readlines()]

In [20]:
channel_names

['CD20',
 'CD31',
 'CD8',
 'HLADRDPDQ',
 'SMA',
 'Tryptase',
 'CD14',
 'Ki67',
 'TOX',
 'ECAD',
 'CK5',
 'GLUT1',
 'CK7',
 'CD68',
 'CD4',
 'gH2AX',
 'Pan_Keratin',
 'CD38',
 'COL1A1',
 'CD3',
 'CD15',
 'FAP']

In [37]:
quant.columns

Index(['pERK', 'FOXP3', 'LaminB1', 'pSTAT1', 'IBA1', 'MHCII', 'CD57', 'CD8a',
       'SMA', 'CD11c', 'yH2AX', 'TAZ', 'Desmin', 'oldCD11b', 'CD31', 'Ki67',
       'TIM3', 'CD4', 'CD20', 'CK7', 'CD45RO', 'MHCI', 'CD3d', 'CD163',
       'CD207', 'HE4', 'Vimentin', 'SNAT1', 'Annexin', 'Ecadherin', 'FOXOA3',
       'KRAS', 'x', 'y', 'sample_id', 'Patient_code_final',
       'Sample_code_final', 'cell_type'],
      dtype='object')

In [39]:
quant

,pERK,FOXP3,LaminB1,pSTAT1,IBA1,MHCII,CD57,CD8a,SMA,CD11c,...,Annexin,Ecadherin,FOXOA3,KRAS,x,y,sample_id,Patient_code_final,Sample_code_final,cell_type
0,0.372753,0.337156,0.367110,0.405686,0.408141,0.486107,0.385586,0.366450,0.292237,0.363262,...,0.400954,0.356802,0.364615,0.398076,2387,6443,20284,S013,S013_primary,IBA1.CD163.Macrophages
1,0.401036,0.509924,0.512477,0.480357,0.512132,0.634644,0.512915,0.511404,0.505456,0.508607,...,0.409148,0.512238,0.512377,0.511586,3426,7956,43273,S013,S013_primary,Proliferating.epithelial
2,0.361901,0.503882,0.504551,0.437978,0.505056,0.396402,0.505478,0.504477,0.505909,0.503802,...,0.442016,0.504303,0.505878,0.504723,1133,5112,562,S013,S013_primary,Fibroblast
3,0.368904,0.536897,0.545607,0.423388,0.542831,0.629951,0.544818,0.538702,0.528172,0.536701,...,0.375871,0.543347,0.544746,0.539554,2746,7921,29026,S013,S013_primary,Proliferating.EMT
4,0.388265,0.501220,0.502478,0.415093,0.502006,0.516949,0.502401,0.501875,0.416413,0.500364,...,0.364061,0.501650,0.501756,0.500740,4891,2644,67527,S013,S013_primary,Proliferating.EMT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6422537,0.461307,0.000763,0.001621,0.518292,0.001122,0.228771,0.000895,0.001016,0.000809,0.001237,...,0.528250,0.001104,0.001428,0.001950,4097,7476,36244,S010,S010_primary,Epithelial
6422538,0.377578,0.000539,0.001472,0.436890,0.000676,0.203623,0.000662,0.000909,0.001035,0.001010,...,0.506569,0.001216,0.001379,0.001910,5185,7323,44801,S010,S010_primary,Epithelial
6422539,0.485293,0.000816,0.001623,0.530833,0.001099,0.223898,0.001038,0.001117,0.001232,0.001130,...,0.565752,0.001024,0.001482,0.002103,2624,7092,17102,S010,S010_primary,Proliferating.epithelial
6422540,0.434089,0.000658,0.001377,0.477240,0.001061,0.209598,0.000784,0.001857,0.001966,0.001138,...,0.673068,0.000535,0.001446,0.001910,4855,5271,42468,S010,S010_primary,CD8.T.cells


In [32]:
X_columns = quant.columns[:quant.columns.get_loc('x')]
obs_columns = quant.columns[quant.columns.get_loc('x'):]
adata = ad.AnnData(
    X=quant[X_columns],
    obs=quant[obs_columns],
    var=pd.DataFrame(index=X_columns)
)
adata.uns['all_markers'] = channel_names

/Users/lukashat/miniforge3/envs/scimap_benchmark/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning:

Transforming to str index.



In [37]:
adata

AnnData object with n_obs × n_vars = 553121 × 40
    obs: 'sample_id', 'cell_id', 'x', 'y', 'Area', 'MajorAxisLength', 'MinorAxisLength', 'Area_Description', 'PatientID', 'RoiID', 'unique_cell_id', 'celltype', 'cell_subtype', 'level_1_cell_type', 'level_2_cell_type', 'cell_type'
    uns: 'all_markers'

In [33]:
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

In [59]:
adata.obs

,area,cell_size,eccentricity,major_axis_length,minor_axis_length,perimeter,sample_id,x,y,area_nuclear,nucleated,overlap_decidua_only,cell_id,level_1_cell_type,level_2_cell_type,cell_type
0,495,495,0.924762,44.126587,16.792203,113.154329,10_31742_1_2,35,1146,0,0,1,1,Immune,Myeloid_immune,Macrophage
1,356,356,0.531811,24.749547,20.959482,81.633514,10_31742_1_2,629,1245,205,1,0,2,unedfined,undefined,undefined
2,325,325,0.762148,27.159477,17.583114,78.183766,10_31742_1_2,1189,760,213,1,1,3,Immune,Myeloid_immune,Macrophage
3,462,462,0.193447,26.663154,26.159504,96.769553,10_31742_1_2,1914,1151,269,1,1,4,Immune,Myeloid_immune,Macrophage
4,721,721,0.870630,45.023270,22.148702,119.047727,10_31742_1_2,322,10,0,0,1,5,unedfined,undefined,undefined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495344,325,325,0.918495,32.838411,12.985409,76.183766,16_31766_3_2,1846,1735,0,0,1,2488,Stromal,Fibroblast,Fibroblast
495345,149,149,0.577342,15.329358,12.516453,45.384776,16_31766_3_2,1718,796,159,1,1,2489,Stromal,Fibroblast,Fibroblast
495346,1123,1123,0.808436,52.251118,30.754148,155.982756,16_31766_3_2,1977,472,0,0,1,2490,Immune,Lymphoid_immune,NK_cell
495347,1205,1205,0.843689,54.493749,29.253976,141.053824,16_31766_3_2,431,749,0,0,1,2491,Stromal,Fibroblast,Fibroblast


In [34]:
adata[adata.obs['sample_id'] == subset].obs

,x,y,Area,MajorAxisLength,MinorAxisLength,Eccentricity,Solidity,Extent,Orientation,cell_id,sample_id,cell_type
330,104.907937,36.552381,630.0,36.215136,23.252212,0.766656,0.913043,0.620690,1.026783,1737,B-12,Cancer
331,365.899844,35.830986,639.0,39.000619,23.456801,0.798913,0.841897,0.570536,1.037478,1738,B-12,Macrophage
332,235.662592,37.180929,818.0,44.250530,24.436838,0.833686,0.916013,0.715035,-1.465577,1745,B-12,Cancer
333,315.258333,41.203333,600.0,43.242619,18.737482,0.901245,0.869565,0.680272,1.501230,1757,B-12,Cancer
334,70.034632,42.000000,231.0,17.603081,16.892832,0.281190,0.962500,0.712963,-1.567969,1762,B-12,Cancer
...,...,...,...,...,...,...,...,...,...,...,...,...
649,247.956364,454.003636,275.0,21.948619,16.733260,0.647126,0.951557,0.770308,0.141479,3261,B-12,Macrophage
650,262.685185,453.162963,270.0,19.693844,18.135529,0.389863,0.957447,0.789474,-0.842525,3262,B-12,CD8+_T_cell
651,411.248387,455.093548,310.0,27.056557,14.856825,0.835755,0.933735,0.615079,1.024074,3267,B-12,CD8+_T_cell
652,445.905325,454.804734,169.0,16.704976,12.950557,0.631653,0.944134,0.704167,-0.676518,3268,B-12,Cancer


In [35]:
sm.pl.image_viewer(image_path=ome_path,adata=adata,x_coordinate='x',y_coordinate='y', point_size=5,imageid='sample_id', subset=subset, overlay='cell_type', seg_mask=seg_path)

In [29]:
subset

'18_31784_5_12'

In [25]:
seg_path

'../../../../../../../../sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/feto_maternal/masks/Segmentation/12_31754_1_14_segmentation_labels.tiff'